# Final exercise: querying the GDELT DOC 2.0 API

This notebook accompanies the slide deck *Accessing Web APIs with Python*. It guides the final exercise in five steps. Each step contains an explanation, a cell to complete (marked `TODO`), and a check cell.

**Tools:** `os`, `time`, `json`, `wget`, `urllib.parse`. Pandas is not used.

**References:** [GDELT DOC 2.0 API documentation](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/) · [GDELT Project](https://www.gdeltproject.org/) · [JSON (Wikipedia)](https://en.wikipedia.org/wiki/JSON)

| Step | Task |
|---|---|
| 0 | Set up the environment and the helper functions |
| 1 | Run two queries and save each response in its own file |
| 2 | Print the five most recent titles of each, with the domain |
| 3 | Count the articles per language |
| 4 | Provoke a failure and handle it with `try`/`except` |
| 5 | Compare `timespan="1d"` and `timespan="1week"` |

## Step 0. Setup

### 0.1 Install the `wget` module

Run the next cell once. If the module is already installed, nothing changes.

In [ ]:
%pip install wget

### 0.2 Imports and working folder

All downloaded files are stored in a `data` folder, created with `os.makedirs`. The option `exist_ok=True` prevents an error if the folder already exists.

In [ ]:
import os
import time
import json
import wget
from urllib.parse import urlencode
from urllib.error import HTTPError, URLError

BASE = "https://api.gdeltproject.org/api/v2/doc/doc"
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)
print("Working folder:", os.getcwd())
print("Data folder exists:", os.path.isdir(DATA_DIR))

### 0.3 Helper functions (provided)

The functions below implement the steps seen in the slides. Read them before continuing; they are not to be modified.

- `build_url` builds the request URL with `urlencode`
- `download` fetches a URL into a file, but only if the file does not already exist, and waits five seconds before each request, as requested by GDELT
- `load_articles` reads the JSON file and removes it if it is empty or invalid, so that a failure is not cached
- `gdelt_search` combines the three

In [ ]:
def build_url(query, mode="ArtList", maxrecords=20, timespan="1week"):
    params = {"query": query, "mode": mode, "maxrecords": maxrecords,
              "timespan": timespan, "format": "json"}
    return BASE + "?" + urlencode(params)


def download(url, filename, pause=5):
    """Download url into filename unless the file already exists."""
    if os.path.exists(filename):
        print("Using cached file:", filename)
        return filename
    time.sleep(pause)                      # politeness towards the server
    wget.download(url, out=filename)
    print()                                # wget's progress bar does not end the line
    return filename


def load_articles(filename):
    """Return the list of articles stored in filename."""
    if os.path.getsize(filename) == 0:
        os.remove(filename)
        return []
    try:
        with open(filename) as f:
            data = json.load(f)
    except ValueError:                     # includes json.JSONDecodeError
        os.remove(filename)
        raise
    return data.get("articles", [])


def gdelt_search(query, filename, maxrecords=20, timespan="1week"):
    url = build_url(query, maxrecords=maxrecords, timespan=timespan)
    download(url, filename)
    return load_articles(filename)

### 0.4 Test the setup

The cell below only builds a URL: it does not contact the server. Check that the output is a single line starting with `https://api.gdeltproject.org/`.

In [ ]:
print(build_url("migration", maxrecords=5, timespan="1d"))

## Step 1. Two queries, two files

**Task.** Choose two topics and save the response of each query in its own file inside `data/`.

**Guidance**

- Use short, specific terms; an exact phrase needs double quotes inside the string, for example `'"climate change"'`
- Add `sourcelang:english` to restrict the language and obtain comparable results
- Use different file names for the two topics
- Each call waits five seconds, so the cell may take a few seconds to complete

In [ ]:
TOPIC_A = "..."   # TODO: first topic, e.g. "migration"
TOPIC_B = "..."   # TODO: second topic

file_a = os.path.join(DATA_DIR, "topic_a.json")
file_b = os.path.join(DATA_DIR, "topic_b.json")

articles_a = []   # TODO: replace with a call to gdelt_search(TOPIC_A, file_a)
articles_b = []   # TODO: replace with a call to gdelt_search(TOPIC_B, file_b)

**Check.**

In [ ]:
assert TOPIC_A != "..." and TOPIC_B != "...", "Choose the two topics first."
assert TOPIC_A != TOPIC_B, "The two topics must differ."
assert os.path.exists(file_a) and os.path.exists(file_b), "Both files must exist in data/."
print(TOPIC_A, "->", len(articles_a), "articles,", os.path.getsize(file_a), "bytes")
print(TOPIC_B, "->", len(articles_b), "articles,", os.path.getsize(file_b), "bytes")

### Inspect the structure

Before writing analysis code, inspect a response. Print the keys of the first article of the first list. If the list is empty, widen the query or the `timespan`.

In [ ]:
if articles_a:
    print(sorted(articles_a[0].keys()))

## Step 2. The five most recent titles

**Task.** For each topic, print the five most recent articles: date, domain and title.

**Guidance**

- The field `seendate` has the form `20261108T120000Z`, so sorting the strings sorts chronologically
- `sorted(list, key=..., reverse=True)` returns a new list; a slice `[:5]` keeps the first five
- Use `a["seendate"]`, `a["domain"]` and `a["title"]`

In [ ]:
def print_recent(articles, n=5):
    """Print the n most recent articles: date | domain, then the title."""
    # TODO: sort the articles by seendate (most recent first)
    # TODO: loop over the first n and print the three fields
    pass

print("Topic A:", TOPIC_A)
print_recent(articles_a)
print()
print("Topic B:", TOPIC_B)
print_recent(articles_b)

**Check.** The function must not modify the original list.

In [ ]:
before = [a["url"] for a in articles_a]
print_recent(articles_a)
assert before == [a["url"] for a in articles_a], "print_recent must not reorder the original list."
print("OK")

## Step 3. Articles per language

**Task.** Write a function `count_by(articles, field)` that returns a dictionary mapping each value of `field` to the number of articles. Use it to count articles per `language` for each topic.

**Guidance**

- Start from an empty dictionary
- `counts.get(value, 0) + 1` avoids a `KeyError` for new values
- Use `a.get(field, "unknown")` in case the field is missing

In [ ]:
def count_by(articles, field):
    counts = {}
    # TODO: fill the dictionary
    return counts

def print_counts(counts):
    for key, n in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
        print(f"{key:25s} {n}")

print("Topic A:", TOPIC_A)
print_counts(count_by(articles_a, "language"))
print()
print("Topic B:", TOPIC_B)
print_counts(count_by(articles_b, "language"))

**Check.** The total of the counts must equal the number of articles.

In [ ]:
for arts in (articles_a, articles_b):
    assert sum(count_by(arts, "language").values()) == len(arts), "Counts do not add up."
print("OK")

**Optional.** Repeat with the field `sourcecountry`. Which countries publish the most articles on each topic?

## Step 4. Provoking and handling failures

**Task.** Reproduce two failures on purpose and handle each one with `try`/`except`.

| Failure | How to provoke it | Expected exception |
|---|---|---|
| Unreachable server | misspell the host name | `URLError` |
| Invalid body | use a query that is too short, such as `"a"`; the server then usually replies with a plain-text message instead of JSON | `ValueError` |

`HTTPError` is a subclass of `URLError`, so it must be listed first.

### 4.1 Unreachable server

In [ ]:
bad_url = build_url("migration").replace("api.gdeltproject.org", "api.gdeltprojekt.org")
bad_file = os.path.join(DATA_DIR, "bad_host.json")

try:
    # TODO: call download(bad_url, bad_file, pause=0)
    pass
except HTTPError as e:
    print("The server replied with status", e.code)
except URLError as e:
    print("The server could not be reached:", e.reason)

### 4.2 Invalid body

In [ ]:
short_file = os.path.join(DATA_DIR, "short_query.json")

try:
    # TODO: call gdelt_search("a", short_file)
    pass
except ValueError:
    print("The response was not valid JSON; the file was removed.")

print("File left on disk:", os.path.exists(short_file))

**Question.** Why does `load_articles` remove the file when the body is invalid? What would happen on the next run if it did not?

## Step 5. Comparing time windows

**Task.** Run the same query with `timespan="1d"` and `timespan="1week"` and compare the numbers of articles and the countries represented.

**Guidance**

- Use the same query and the same `maxrecords` for both runs, otherwise the comparison is not meaningful
- Because `maxrecords` caps the list, a high value such as `250` is needed to see differences
- Save the two responses in two files

In [ ]:
QUERY = TOPIC_A
file_1d = os.path.join(DATA_DIR, "window_1d.json")
file_1w = os.path.join(DATA_DIR, "window_1week.json")

arts_1d = []   # TODO: gdelt_search(QUERY, file_1d, maxrecords=250, timespan="1d")
arts_1w = []   # TODO: gdelt_search(QUERY, file_1w, maxrecords=250, timespan="1week")

print("1 day :", len(arts_1d), "articles,", len(count_by(arts_1d, "sourcecountry")), "countries")
print("1 week:", len(arts_1w), "articles,", len(count_by(arts_1w, "sourcecountry")), "countries")

**Questions**

1. Is the number of articles for one week seven times that for one day? Why, or why not?
2. Does the list of the most frequent countries change between the two windows?
3. What limits the number of articles returned, regardless of the time window?

## Wrap-up

Before submitting, check that:

- [ ] the `data/` folder contains the saved responses and no empty or invalid file
- [ ] every `TODO` has been replaced
- [ ] the notebook runs from top to bottom (*Kernel → Restart and run all*); the second run should use the cached files and take only a moment
- [ ] you can explain, in one sentence each, what `urlencode`, `wget`, `os.path.exists` and `json.load` do in this notebook

**Further reading:** [GDELT DOC 2.0 API documentation](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/) · [HTTP status codes (Wikipedia)](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes)